In [5]:
require_relative 'draw_dot'

true

In [7]:
require 'set'

class Value
    attr_reader :data, :children, :op
    attr_accessor :label, :grad, :backward

    def initialize(data, children = [], operator = '', label: '')
        @data = data
        @grad = 0.0
        @backward = -> {}
        @children = children.to_set # consider other data structures?
        @op = operator
        @label = label
    end

    def inspect
        # similar to python just for the lulz
        "Value:(data=#{@data})"
    end

    def +(operand)
        operand = ensure_value(operand)

        out = Value.new( self.data + operand.data, [self, operand], '+')
        out.backward = lambda {
            self.grad += 1.0 * out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
            operand.grad += 1.0 *out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
        }
        out
    end

    
    def *(operand)
        operand = ensure_value(operand)
        
        out = Value.new( self.data * operand.data, [self, operand], '*')
        out.backward = lambda {
            self.grad += operand.data * out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
            operand.grad += self.data * out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
        }
        out
    end

    def coerce(other) # this is fow handling non-Value-types i.e '1 + Value'
      [ensure_value(other), self]
    end

    def -@
      self * -1
    end

    def -(other)
      self + (-ensure_value(other))
    end

    def /(other)
      self * (ensure_value(other)**-1)
    end

    def **(other)
      out = Value.new(self.data**ensure_value(other), [self], "**#{other}")

      out.backward = lambda {
        self.grad += (other * (@data**(other - 1))) * out.grad
      }

      out
    end

    # activation (squashing) function
    def tanh
        x = self.data
        t = (Math.exp(2 * x) - 1) / (Math.exp(2 * x) + 1)
        out = Value.new(t, [self], 'tanh - squashing/activation func')

        out.backward = lambda {
            self.grad = (1 - out.data**2) * out.grad
        }
        out
    end

    
    def propagate_all # manual back propagation to calculate gradients (grad)
      topo = []
      visited = Set.new

      build_topo = lambda do |v|
        next if visited.include?(v)

        visited.add(v)
        v.children.each { |child| build_topo.call(child) }
        topo << v
      end

      build_topo.call(self)

      @grad = 1.0
      topo.reverse_each { |v| v.backward.call }
    end

    private

    def ensure_value(operand)
        operand =   
        case operand
        when Value then operand
        when Integer, Float then Value.new(operand)
        else raise("Hey, '#{operand}' must be and Integer or Float")
        end
    end
    
end


:ensure_value

In [8]:
class Neuron
    def initialize(number_of_inputs_for_a_neuron)
        @w = Array.new(number_of_inputs_for_a_neuron) { Value.new(rand(-1.0..1.0), label: 'w') }
        @b = Value.new(rand(-1.0..1.0), label: 'b')
    end

    def call(x)
        nuron_body_for_activation = 
            @w
            .zip(x)
            .map {  |wi, xi|  wi * xi  }
            .sum + @b
            
        nuron_body_for_activation.tanh
    end
end

class Layer
    def initialize(number_of_inputs_for_a_neuron, number_of_layer_neuron_outputs)
      @neurons = Array.new(number_of_layer_neuron_outputs) do
          Neuron.new(number_of_inputs_for_a_neuron) 
      end
    end

    def call(x)
      out = @neurons.map { |neuron| neuron.call(x) }
      out.length == 1 ? out[0] : out
    end
end

class MultiLayerPerceptron
    def initialize(nin, nouts_list)
      sz = [nin] + nouts_list
      @layers = nouts_list.each_index.map do |i|
        Layer.new(sz[i], sz[i + 1])
      end
    end

    def call(x)
      @layers.each { |layer| x = layer.call(x) }
      x
    end
end

list_of_signals_x = [2.0, 3.0, -1.0]

# refer pic above. we build perceptron for hidden and output layers, and call in with a list of signals fron the input layer
# input_layer_size = 3 # og list_of_signals_x.length
# hidden_layer_1_size = 4
# hidden_layer_2_size = 4
# output_layer_size = 1
mlp = MultiLayerPerceptron.new(3, [4, 4, 1])

mlp.call(list_of_signals_x)

Value:(data=0.6169014580212419)

In [9]:
mlp_graph = draw_dot_graphviz(mlp.call(list_of_signals_x))
mlp_graph.output(svg: "mlp_graph2.svg")